# Kev Data Collection on Grummon et al. (2025) FOP Nutrition Label RCT (CUDA)

In [1]:
# Step 1: Load Grummon et al. (2025) RCT dataset and subsample the 200 participants (1,200 choice tasks, seed=42)
import importlib
from pathlib import Path
import torch
import grummon_ols
import kev_eval

importlib.reload(kev_eval)
importlib.reload(grummon_ols)
from grummon_ols import load_grummon_rct, subsample_grummon_participants
from kev_eval import (
    SEEN_PERSONA_COLS,
    UNSEEN_PERSONA_COLS,
    build_kev_prompt_items,
    collect_kev_prompts,
    collect_kev_single_attr_prompts,
    free_kev_memory,
    load_kev_model,
    preview_kev_item,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_LLM_PARTICIPANTS = 200
SUBSAMPLE_SEED = 42
DATA_DIR = Path("data")
ABLATION_DIR = DATA_DIR / "persona_ablation"

raw_df = load_grummon_rct()
sub_df = subsample_grummon_participants(raw_df, n_participants=N_LLM_PARTICIPANTS, seed=SUBSAMPLE_SEED)
print(
    f"Device: {DEVICE} | Subsampled {sub_df['CASEID'].nunique()} participants ({len(sub_df):,} tasks) "
    f"across {sub_df['arm'].nunique()} arms | Seen attrs={len(SEEN_PERSONA_COLS)}, Unseen attrs={len(UNSEEN_PERSONA_COLS)}"
)

Device: cuda | Subsampled 200 participants (1,200 tasks) across 6 arms | Seen attrs=5, Unseen attrs=5


In [2]:
# Step 2: Build Kev prompt items for Random, Full Respondent (10 attrs), Seen (5 attrs), and Unseen (5 attrs)
kev_items_random = build_kev_prompt_items(sub_df, persona_mode="random", include_country=False)
kev_items_persona = build_kev_prompt_items(sub_df, persona_mode="persona", include_country=False)
kev_items_seen = build_kev_prompt_items(sub_df, persona_mode="seen", include_country=False)
kev_items_unseen = build_kev_prompt_items(sub_df, persona_mode="unseen", include_country=False)

print("=== Sample Random Persona Kev Prompt ===")
print(preview_kev_item(kev_items_random[0]))
print("\n=== Sample Respondent Persona Kev Prompt (10 attributes) ===")
print(preview_kev_item(kev_items_persona[0]))

=== Sample Random Persona Kev Prompt ===


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

70 tokens; state 1 tokens (no truncation, strict=True)
<|fim_prefix|>
<|fim_middle|>
Click on the product you think is healthier, or click 'They're the same' if you think the two products are equally healthy.
<|box_start|>Ocean Spray 100% Cranberry juice: Front-of-package label: None<|box_end|>
<|box_start|>Ocean Spray Cranberry Juice Cocktail: Front-of-package label: None<|box_end|>
<|box_start|>They're the same<|box_end|>
<|fim_suffix|>

=== Sample Respondent Persona Kev Prompt (10 attributes) ===


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

166 tokens; state 97 tokens (no truncation, strict=True)
<|fim_prefix|>
Sex: Female
Age: 30-49
Education: College graduate/some postgrad
Income: $50,000 - $75,000
Race: White
Hispanic or Latino: No
SNAP / Food Stamps recipient: No
Nutrition Facts label usage: Most of the time
Share of groceries bought online: Less than 25%
Nutrition literacy score: 4 out of 4
<|fim_middle|>
Click on the product you think is healthier, or click 'They're the same' if you think the two products are equally healthy.
<|box_start|>Ocean Spray 100% Cranberry juice: Front-of-package label: None<|box_end|>
<|box_start|>Ocean Spray Cranberry Juice Cocktail: Front-of-package label: None<|box_end|>
<|box_start|>They're the same<|box_end|>
<|fim_suffix|>


In [3]:
# Step 3: Collect Kev 0.8B Base & Fine-Tuned (newvalid/best) predictions on CUDA
MODEL_SPECS_08B = [
    ("jaredpalmer/kev-0.8b", "jaredpalmer/kev-0.8b", "kev-0.8b-base"),
    ("../../runs/kev-0.8b-resp-issp+twin2k-grouped-newvalid/best", "kev-0.8b-resp-issp+twin2k-grouped-newvalid/best", "kev-0.8b-resp-issp+twin2k-grouped-newvalid_best"),
]

for ckpt_path, model_name, slug in MODEL_SPECS_08B:
    model = None
    free_kev_memory()
    model = load_kev_model(ckpt_path, device=DEVICE, temperature=None)
    try:
        collect_kev_prompts(model, kev_items_random, DATA_DIR / f"{slug}_random.jsonl", model_name=model_name)
        collect_kev_prompts(model, kev_items_persona, DATA_DIR / f"{slug}_persona.jsonl", model_name=model_name)
        collect_kev_prompts(model, kev_items_seen, ABLATION_DIR / f"{slug}_seen.jsonl", model_name=model_name)
        collect_kev_prompts(model, kev_items_unseen, ABLATION_DIR / f"{slug}_unseen.jsonl", model_name=model_name)
        collect_kev_single_attr_prompts(model, sub_df, ABLATION_DIR / f"{slug}_single_attrs.jsonl", model_name=model_name)
    finally:
        model = None
        free_kev_memory()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[kev-0.8b-base_random.jsonl] Complete: all 1,200/1,200 requested items already saved (total in file: 1,200).
[kev-0.8b-base_persona.jsonl] Complete: all 1,200/1,200 requested items already saved (total in file: 1,200).
[kev-0.8b-base_seen.jsonl] Collecting 1,200 missing items across 174 unique (state, arm) groups on device=cuda (model='jaredpalmer/kev-0.8b', T=2.3511)...


kev-0.8b-base_seen (cuda):   0%|          | 0/174 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


... (11 outputs left out)

[kev-0.8b-resp-issp+twin2k-grouped-newvalid_best_unseen.jsonl] Collecting 1,200 missing items across 151 unique (state, arm) groups on device=cuda (model='kev-0.8b-resp-issp+twin2k-grouped-newvalid/best', T=1.0000)...


kev-0.8b-resp-issp+twin2k-grouped-newvalid_best_unseen (cuda):   0%|          | 0/151 [00:00<?, ?it/s]

[kev-0.8b-resp-issp+twin2k-grouped-newvalid_best_single_attrs.jsonl] Collecting 11,928 single-attribute items across 216 unique (state, arm) groups on device=cuda (model='kev-0.8b-resp-issp+twin2k-grouped-newvalid/best', T=1.0000)...


kev-0.8b-resp-issp+twin2k-grouped-newvalid_best_single_attrs (cuda):   0%|          | 0/216 [00:00<?, ?it/s]

In [4]:
# Step 4: Collect Kev 4B Base & Fine-Tuned (newvalid/best) predictions on CUDA
MODEL_SPECS_4B = [
    ("jaredpalmer/kev-4b", "jaredpalmer/kev-4b", "kev-4b-base"),
    ("../../runs/kev-4b-resp-issp+twin2k-grouped-newvalid/best", "kev-4b-resp-issp+twin2k-grouped-newvalid/best", "kev-4b-resp-issp+twin2k-grouped-newvalid_best"),
]

for ckpt_path, model_name, slug in MODEL_SPECS_4B:
    model = None
    free_kev_memory()
    model = load_kev_model(ckpt_path, device=DEVICE, temperature=None)
    try:
        collect_kev_prompts(model, kev_items_random, DATA_DIR / f"{slug}_random.jsonl", model_name=model_name)
        collect_kev_prompts(model, kev_items_persona, DATA_DIR / f"{slug}_persona.jsonl", model_name=model_name)
        collect_kev_prompts(model, kev_items_seen, ABLATION_DIR / f"{slug}_seen.jsonl", model_name=model_name)
        collect_kev_prompts(model, kev_items_unseen, ABLATION_DIR / f"{slug}_unseen.jsonl", model_name=model_name)
        collect_kev_single_attr_prompts(model, sub_df, ABLATION_DIR / f"{slug}_single_attrs.jsonl", model_name=model_name)
    finally:
        model = None
        free_kev_memory()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

[kev-4b-base_random.jsonl] Collecting 1,200 missing items across 6 unique (state, arm) groups on device=cuda (model='jaredpalmer/kev-4b', T=2.4061)...


... (18 outputs left out)

[kev-4b-resp-issp+twin2k-grouped-newvalid_best_unseen.jsonl] Collecting 1,200 missing items across 151 unique (state, arm) groups on device=cuda (model='kev-4b-resp-issp+twin2k-grouped-newvalid/best', T=1.0000)...


kev-4b-resp-issp+twin2k-grouped-newvalid_best_unseen (cuda):   0%|          | 0/151 [00:00<?, ?it/s]

[kev-4b-resp-issp+twin2k-grouped-newvalid_best_single_attrs.jsonl] Collecting 11,928 single-attribute items across 216 unique (state, arm) groups on device=cuda (model='kev-4b-resp-issp+twin2k-grouped-newvalid/best', T=1.0000)...


kev-4b-resp-issp+twin2k-grouped-newvalid_best_single_attrs (cuda):   0%|          | 0/216 [00:00<?, ?it/s]

In [5]:
# Step 5: Collect Kev 9B Base & Fine-Tuned (newvalid/best) predictions on CUDA
MODEL_SPECS_9B = [
    ("jaredpalmer/kev-9b", "jaredpalmer/kev-9b", "kev-9b-base"),
    ("../../runs/kev-9b-resp-issp+twin2k-grouped-newvalid/best", "kev-9b-resp-issp+twin2k-grouped-newvalid/best", "kev-9b-resp-issp+twin2k-grouped-newvalid_best"),
]

for ckpt_path, model_name, slug in MODEL_SPECS_9B:
    model = None
    free_kev_memory()
    model = load_kev_model(ckpt_path, device=DEVICE, temperature=None)
    try:
        collect_kev_prompts(model, kev_items_random, DATA_DIR / f"{slug}_random.jsonl", model_name=model_name)
        collect_kev_prompts(model, kev_items_persona, DATA_DIR / f"{slug}_persona.jsonl", model_name=model_name)
        collect_kev_prompts(model, kev_items_seen, ABLATION_DIR / f"{slug}_seen.jsonl", model_name=model_name)
        collect_kev_prompts(model, kev_items_unseen, ABLATION_DIR / f"{slug}_unseen.jsonl", model_name=model_name)
        collect_kev_single_attr_prompts(model, sub_df, ABLATION_DIR / f"{slug}_single_attrs.jsonl", model_name=model_name)
    finally:
        model = None
        free_kev_memory()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

[kev-9b-base_random.jsonl] Complete: all 1,200/1,200 requested items already saved (total in file: 1,200).
[kev-9b-base_persona.jsonl] Complete: all 1,200/1,200 requested items already saved (total in file: 1,200).
[kev-9b-base_seen.jsonl] Collecting 1,200 missing items across 174 unique (state, arm) groups on device=cuda (model='jaredpalmer/kev-9b', T=2.1936)...


... (10 outputs left out)

[kev-9b-resp-issp+twin2k-grouped-newvalid_best_unseen.jsonl] Collecting 1,200 missing items across 151 unique (state, arm) groups on device=cuda (model='kev-9b-resp-issp+twin2k-grouped-newvalid/best', T=1.0000)...


kev-9b-resp-issp+twin2k-grouped-newvalid_best_unseen (cuda):   0%|          | 0/151 [00:00<?, ?it/s]

[kev-9b-resp-issp+twin2k-grouped-newvalid_best_single_attrs.jsonl] Collecting 11,928 single-attribute items across 216 unique (state, arm) groups on device=cuda (model='kev-9b-resp-issp+twin2k-grouped-newvalid/best', T=1.0000)...


kev-9b-resp-issp+twin2k-grouped-newvalid_best_single_attrs (cuda):   0%|          | 0/216 [00:00<?, ?it/s]